# Laboratorio 9. Árboles de decisión

## Cargar y explorar los datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

print("Dimensiones:", X.shape)
print("Clases:", data.target_names)

display(X.head())
print(y.value_counts())

## Dividir en entrenamiento y prueba

Utilizaremos 80% de las observaciones para entrenamiento y 20% para prueba.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Entrenar nuestro primer árbol

Comenzamos con un árbol de profundidad máxima 3.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy entrenamiento:", model.score(X_train, y_train))
print("Accuracy prueba:", accuracy_score(y_test, y_pred))

## Visualizar el árbol

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 9))

plot_tree(
    model,
    feature_names=X.columns,
    class_names=list(data.target_names),
    filled=True,
    rounded=True,
    fontsize=9
)

plt.show()

Aquí podemos identificar las variables seleccionadas, los puntos de corte, la impureza Gini y las predicciones de las hojas.

## Experimentar con diferentes profundidades

Ahora entrenaremos varios árboles para observar cómo cambia su desempeño.

In [ ]:
depths = range(1, 16)

train_acc = []
test_acc = []

for depth in depths:
    
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    train_acc.append(model.score(X_train, y_train))
    test_acc.append(model.score(X_test, y_test))

plt.figure(figsize=(9, 5))

plt.plot(depths, train_acc, marker="o", label="Entrenamiento")
plt.plot(depths, test_acc, marker="o", label="Prueba")

plt.xlabel("Profundidad máxima")
plt.ylabel("Accuracy")
plt.title("Efecto de la profundidad del árbol")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Comparar diferentes hiperparámetros
Además de max_depth, podemos modificar:


| Hiperparámetro | Descripción |
|---|---|
| `criterion` | Medida de impureza utilizada para seleccionar divisiones (`gini`, `entropy`, `log_loss`). |
| `max_depth` | Profundidad máxima permitida del árbol. |
| `min_samples_split` | Número mínimo de observaciones necesarias para dividir un nodo. |
| `min_samples_leaf` | Número mínimo de observaciones que debe contener cada hoja. |
| `max_leaf_nodes` | Número máximo de nodos terminales permitidos. |


Podemos comparar algunas configuraciones mediante validación cruzada:

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [2, 3, 5, 10, None],
    "min_samples_leaf": [1, 3, 5, 10]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train, y_train)

print("Mejores parámetros:", grid.best_params_)
print("Accuracy CV:", grid.best_score_)
print("Accuracy prueba:", grid.score(X_test, y_test))

## Visualizar el efecto de cada hiperparámetro

En lugar de utilizar solamente GridSearchCV, entrenaremos varios modelos modificando un hiperparámetro a la vez.

Compararemos el accuracy de entrenamiento, el accuracy de prueba, la profundidad real y el número de hojas.

In [ ]:
configuraciones = {
    "max_depth": [1, 2, 3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10, 20, 50],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_leaf_nodes": [2, 3, 5, 10, 20, 50],
    "ccp_alpha": [0, 0.001, 0.005, 0.01, 0.02]
}

resultados = []

for parametro, valores in configuraciones.items():
    for valor in valores:

        modelo = DecisionTreeClassifier(
            random_state=42,
            **{parametro: valor}
        )

        modelo.fit(X_train, y_train)

        resultados.append({
            "Parámetro": parametro,
            "Valor": str(valor),
            "Accuracy train": modelo.score(X_train, y_train),
            "Accuracy test": modelo.score(X_test, y_test),
            "Profundidad": modelo.get_depth(),
            "Hojas": modelo.get_n_leaves()
        })

df_resultados = pd.DataFrame(resultados)

display(df_resultados)


Visualizaremos cómo cambia el accuracy al modificar cada configuración.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes = axes.flatten()

for i, (parametro, valores) in enumerate(configuraciones.items()):

    datos = df_resultados[
        df_resultados["Parámetro"] == parametro
    ]

    x = range(len(datos))

    axes[i].plot(
        x, datos["Accuracy train"],
        marker="o", label="Train"
    )

    axes[i].plot(
        x, datos["Accuracy test"],
        marker="o", label="Test"
    )

    axes[i].set_xticks(list(x))
    axes[i].set_xticklabels(datos["Valor"])

    axes[i].set_title(parametro)
    axes[i].set_xlabel("Valor del hiperparámetro")
    axes[i].set_ylabel("Accuracy")
    axes[i].legend()
    axes[i].grid(alpha=0.3)

axes[-1].axis("off")

plt.tight_layout()
plt.show()


Esto permite identificar configuraciones que producen sobreajuste y observar que un árbol más complejo no necesariamente generaliza mejor.

## Comparar los árboles visualmente

In [ ]:

from sklearn.tree import plot_tree

profundidades = [1, 2, 3, 4]

for profundidad in profundidades:

    modelo = DecisionTreeClassifier(
        max_depth=profundidad,
        random_state=42
    )

    modelo.fit(X_train, y_train)

    plt.figure(figsize=(16, 7))

    plot_tree(
        modelo,
        feature_names=X.columns,
        class_names=list(data.target_names),
        filled=True,
        rounded=True,
        fontsize=9
    )

    plt.title(f"Árbol con max_depth = {profundidad}")
    plt.show()
